# Tongsan.org Full Content Scraper

Scrapes ALL content from Tongsan.org (Chin/Zomi news site) using the WordPress REST API.

## What It Scrapes
- **All articles** across all categories (news, articles, education, multimedia, sports, etc.)
- **3 languages**: Zokam (default), English (`/enn/`), Burmese (`/my/`)
- **Categories, tags, authors, media**
- **Full article content** (HTML + metadata)

## Data Sources
- WordPress REST API: `/wp-json/wp/v2/`
- Sitemaps: `/wp-sitemap.xml`
- RSS feeds: `/feed/`

## Output
- `tongsan_articles.jsonl` — all articles with full content
- `tongsan_categories.json` — category tree
- `tongsan_tags.json` — all tags
- `tongsan_summary.json` — stats and metadata

In [ ]:
# Step 0: Setup
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

def check_internet() -> bool:
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

if not check_internet():
    raise SystemError("No internet connection.")
print("Internet: OK")

try:
    import requests
    from bs4 import BeautifulSoup
    print("Packages: OK")
except ImportError as e:
    print(f"Installing: {e}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "beautifulsoup4", "-q"])
    print("Packages installed.")

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"IS_KAGGLE: {IS_KAGGLE}  |  WORK_DIR: {WORK_DIR}")


In [ ]:
# Step 1: Discover site structure via WordPress REST API
import requests as req

BASE_URL = "https://tongsan.org"
API_BASE = f"{BASE_URL}/wp-json/wp/v2"

def api_get(endpoint: str, params: dict | None = None) -> list | dict:
    url = f"{API_BASE}/{endpoint}"
    r = req.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def api_get_all(endpoint: str, per_page: int = 100) -> list:
    """Fetch all pages of a paginated endpoint."""
    all_items = []
    page = 1
    while True:
        r = req.get(f"{API_BASE}/{endpoint}", params={"per_page": per_page, "page": page}, timeout=30)
        r.raise_for_status()
        items = r.json()
        if not items:
            break
        all_items.extend(items)
        total = int(r.headers.get("X-WP-Total", "0"))
        print(f"  {endpoint}: page {page} ({len(items)} items, total so far: {len(all_items)}/{total})")
        if len(all_items) >= total:
            break
        page += 1
        time.sleep(0.3)
    return all_items

# Fetch categories
print("Fetching categories...")
categories = api_get_all("categories")
print(f"Total categories: {len(categories)}")

# Fetch tags
print("Fetching tags...")
tags = api_get_all("tags")
print(f"Total tags: {len(tags)}")

# Fetch users/authors
print("Fetching authors...")
users = api_get("users")
print(f"Total authors: {len(users)}")

# Build category tree
cat_map = {c["id"]: c for c in categories}
for c in categories:
    parent = cat_map.get(c.get("parent"))
    if parent:
        c["_parent_name"] = parent.get("name", "")

# Save categories and tags
cat_path = WORK_DIR / "tongsan_categories.json"
tag_path = WORK_DIR / "tongsan_tags.json"
cat_path.write_text(json.dumps(categories, ensure_ascii=False, indent=2), encoding="utf-8")
tag_path.write_text(json.dumps(tags, ensure_ascii=False, indent=2), encoding="utf-8")

# Print category tree
print("\n=== Category Tree ===")
roots = [c for c in categories if c.get("parent") == 0]
children = [c for c in categories if c.get("parent", 0) != 0]
for root in sorted(roots, key=lambda x: x.get("count", 0), reverse=True):
    print(f"  {root['name']} ({root.get('count', 0)} articles)")
    for child in sorted([c for c in children if c.get("parent") == root["id"]], key=lambda x: x.get("count", 0), reverse=True):
        print(f"    └─ {child['name']} ({child.get('count', 0)} articles)")

# Total post count
total_posts_info = api_get("posts", {"per_page": 1})
print(f"\nTotal articles available: {total_posts_info}")


In [ ]:
# Step 2: Fetch all articles
import concurrent.futures
import threading

ARTICLES_OUTPUT = WORK_DIR / "tongsan_articles.jsonl"
PROGRESS_LOG = WORK_DIR / "tongsan_progress.log"

def _log(msg: str) -> None:
    line = f"{time.strftime('%H:%M:%S')}  {msg}\n"
    sys.stdout.write(line)
    sys.stdout.flush()
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line)

def fetch_article_page(page: int, per_page: int = 100) -> tuple[int, list]:
    r = req.get(f"{API_BASE}/posts", params={"per_page": per_page, "page": page, "_embed": 1}, timeout=60)
    r.raise_for_status()
    return page, r.json()

def fetch_all_articles(max_workers: int = 5, per_page: int = 100) -> int:
    """Fetch all articles using concurrent requests."""
    # Get total count with the SAME per_page we'll use for fetching
    r = req.get(f"{API_BASE}/posts", params={"per_page": per_page}, timeout=30)
    r.raise_for_status()
    total = int(r.headers.get("X-WP-Total", "0"))
    total_pages = int(r.headers.get("X-WP-TotalPages", "0"))
    _log(f"Total articles: {total:,} across {total_pages} pages (per_page={per_page})")

    if total == 0:
        return 0

    # Check which article IDs are already saved (by ID, not page — avoids per_page mismatch)
    done_ids = set()
    if ARTICLES_OUTPUT.exists():
        with open(ARTICLES_OUTPUT, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line.strip())
                    aid = obj.get("id")
                    if aid:
                        done_ids.add(aid)
                except json.JSONDecodeError:
                    pass
    if done_ids:
        _log(f"Resuming: {len(done_ids)} articles already saved")

    pages_to_fetch = list(range(1, total_pages + 1))
    _log(f"Fetching {len(pages_to_fetch)} pages with {max_workers} workers...")

    fetched = 0
    skipped = 0
    write_lock = threading.Lock()
    started_at = time.time()

    mode = "a" if ARTICLES_OUTPUT.exists() else "w"
    with open(ARTICLES_OUTPUT, mode, encoding="utf-8") as out:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_page = {executor.submit(fetch_article_page, p): p for p in pages_to_fetch}
            for future in concurrent.futures.as_completed(future_to_page):
                page = future_to_page[future]
                try:
                    page_num, articles = future.result()
                    for article in articles:
                        aid = article.get("id")
                        if aid in done_ids:
                            skipped += 1
                            continue
                        done_ids.add(aid)
                        cat_names = [cat_map.get(cid, {}).get("name", "") for cid in article.get("categories", [])]
                        tag_names = [t.get("name", "") for t in tags if t.get("id") in article.get("tags", [])]
                        record = {
                            "id": article.get("id"),
                            "date": article.get("date"),
                            "modified": article.get("modified"),
                            "slug": article.get("slug"),
                            "title": article.get("title", {}).get("rendered", ""),
                            "excerpt": article.get("excerpt", {}).get("rendered", ""),
                            "content": article.get("content", {}).get("rendered", ""),
                            "link": article.get("link"),
                            "author": article.get("author"),
                            "categories": article.get("categories", []),
                            "category_names": cat_names,
                            "tags": article.get("tags", []),
                            "tag_names": tag_names,
                            "featured_media": article.get("featured_media"),
                            "_page": page_num,
                        }
                        with write_lock:
                            out.write(json.dumps(record, ensure_ascii=False) + "\n")
                            out.flush()
                            fetched += 1
                    total_processed = fetched + skipped
                    if total_processed % 200 == 0:
                        elapsed = time.time() - started_at
                        rate = total_processed / elapsed if elapsed > 0 else 0
                        _log(f"Processed {total_processed}/{total} articles (new={fetched}, skip={skipped}) rate={rate:.1f}/s")
                    time.sleep(0.3)
                except Exception as e:
                    _log(f"Error fetching page {page}: {e}")

    elapsed = time.time() - started_at
    if fetched == 0 and skipped > 0:
        _log(f"All {skipped} articles already saved. Nothing new to fetch.")
    else:
        _log(f"Done! New={fetched}, skipped={skipped}, total={total} in {elapsed/60:.1f} minutes")
    return fetched + skipped

_log("Starting article fetch...")
count = fetch_all_articles(max_workers=5)
print(f"\nTotal articles saved: {count:,}")


In [ ]:
# Step 3: Language detection — all posts are in the main API
# English and Burmese articles are already included in the main feed.
# Language is detected from the URL path in the 'link' field:
#   /enn/  → English
#   /my/   → Burmese
#   no prefix → Zokam (default)
# No separate fetch needed — Step 2 already got everything.

def detect_lang(link: str) -> str:
    if '/enn/' in link:
        return 'enn'
    if '/my/' in link:
        return 'my'
    return 'zokam'

# Count languages in saved output
lang_counts = {"zokam": 0, "enn": 0, "my": 0}
if ARTICLES_OUTPUT.exists():
    with open(ARTICLES_OUTPUT, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line.strip())
                lang = detect_lang(obj.get("link", ""))
                lang_counts[lang] = lang_counts.get(lang, 0) + 1
            except: pass

print(f"\n=== Language Breakdown ===")
print(f"  Zokam:   {lang_counts.get('zokam', 0):,}")
print(f"  English: {lang_counts.get('enn', 0):,}")
print(f"  Burmese: {lang_counts.get('my', 0):,}")
print(f"  Total:   {sum(lang_counts.values()):,}")


In [ ]:
# Step 4: Add language tags and generate summary
FINAL_OUTPUT = WORK_DIR / "tongsan_all_articles.jsonl"

def detect_lang(link: str) -> str:
    if '/enn/' in link: return 'enn'
    if '/my/' in link: return 'my'
    return 'zokam'

merged = 0
lang_counts = {"zokam": 0, "enn": 0, "my": 0}
cat_article_counts: dict[str, int] = {}

with open(FINAL_OUTPUT, "w", encoding="utf-8") as out:
    with open(ARTICLES_OUTPUT, "r", encoding="utf-8") as inp:
        for line in inp:
            ls = line.strip()
            if not ls: continue
            try:
                obj = json.loads(ls)
            except json.JSONDecodeError: continue
            lang = detect_lang(obj.get("link", ""))
            obj["_source_lang"] = lang
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
            merged += 1
            lang_counts[lang] = lang_counts.get(lang, 0) + 1
            for cn in obj.get("category_names", []):
                cat_article_counts[cn] = cat_article_counts.get(cn, 0) + 1

print(f"\n{'='*50}")
print(f"Results")
print(f"{'='*50}")
print(f"Total articles: {merged:,}")
print(f"  Zokam:   {lang_counts.get('zokam', 0):,}")
print(f"  English: {lang_counts.get('enn', 0):,}")
print(f"  Burmese: {lang_counts.get('my', 0):,}")
print(f"File size: {FINAL_OUTPUT.stat().st_size/1024/1024:.1f} MB")

print(f"\nTop categories:")
for cat, cnt in sorted(cat_article_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  {cat}: {cnt:,}")

summary = {
    "source": "tongsan.org",
    "scraped_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "total_articles": merged,
    "by_language": lang_counts,
    "total_categories": len(categories),
    "total_tags": len(tags),
    "top_categories": dict(sorted(cat_article_counts.items(), key=lambda x: x[1], reverse=True)[:20]),
    "output": str(FINAL_OUTPUT),
    "file_size_mb": round(FINAL_OUTPUT.stat().st_size / 1024 / 1024, 1),
}
summary_path = WORK_DIR / "tongsan_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSummary saved: {summary_path}")
print(json.dumps(summary, ensure_ascii=False, indent=2))


# Output Files

| File | Description |
|------|-------------|
| `tongsan_all_articles.jsonl` | All articles (Zokam + English + Burmese) |
| `tongsan_categories.json` | Full category tree with counts |
| `tongsan_tags.json` | All tags |
| `tongsan_summary.json` | Stats and metadata |

## Article Format (JSONL)
Each line is a JSON object with:
- `id`, `date`, `slug`, `title`, `excerpt`, `content`
- `link` (URL to original article)
- `category_names`, `tag_names`
- `_source_lang` (zokam/enn/my)
